<a href="https://colab.research.google.com/github/anitejpadyana/BIS-lab/blob/main/Particle_Swarm_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np

subjects = ["CN", "AI", "OOM", "DEV", "Adv Algorithm"]

min_hours = np.array([2, 2, 1, 1, 1], dtype=float)
max_hours = np.array([6, 6, 5, 5, 5], dtype=float)

weights = np.array([0.20, 0.20, 0.20, 0.15, 0.25])

total_hours = 15

N = 100
D = len(subjects)

w = 0.7
c1 = 1.5
c2 = 1.5
MaxIter = 500


def repair(x):
    x = np.clip(x, min_hours, max_hours)

    difference = total_hours - np.sum(x)

    while abs(difference) > 1e-10:
        if difference > 0:
            available = max_hours - x
            indices = np.where(available > 1e-10)[0]

            if len(indices) == 0:
                break

            add = min(difference, np.sum(available))
            allocation = available / np.sum(available) * add
            x += allocation

        else:
            available = x - min_hours
            indices = np.where(available > 1e-10)[0]

            if len(indices) == 0:
                break

            remove = min(-difference, np.sum(available))
            allocation = available / np.sum(available) * remove
            x -= allocation

        difference = total_hours - np.sum(x)

    return x


def fitness(x):
    return np.sum(weights * x)


x = np.random.uniform(min_hours, max_hours, (N, D))

for i in range(N):
    x[i] = repair(x[i])

v = np.random.uniform(
    -(max_hours - min_hours),
    (max_hours - min_hours),
    (N, D)
)

pbest = x.copy()
pbest_fit = np.array([fitness(pbest[i]) for i in range(N)])

best_index = np.argmax(pbest_fit)

gbest = pbest[best_index].copy()
gbest_fit = pbest_fit[best_index]

for iteration in range(MaxIter):

    for i in range(N):

        r1 = np.random.random(D)
        r2 = np.random.random(D)

        v[i] = (
            w * v[i]
            + c1 * r1 * (pbest[i] - x[i])
            + c2 * r2 * (gbest - x[i])
        )

        x[i] = x[i] + v[i]

        x[i] = np.clip(x[i], min_hours, max_hours)

        x[i] = repair(x[i])

        fit = fitness(x[i])

        if fit > pbest_fit[i]:
            pbest[i] = x[i].copy()
            pbest_fit[i] = fit

        if fit > gbest_fit:
            gbest = x[i].copy()
            gbest_fit = fit

print("Best Study-Time Allocation")
print("-" * 35)

for i in range(D):
    print(f"{subjects[i]:20s}: {gbest[i]:.2f} hours")

print("-" * 35)
print(f"Total hours           : {np.sum(gbest):.2f}")
print(f"Best fitness          : {gbest_fit:.4f}")

Best Study-Time Allocation
-----------------------------------
CN                  : 2.00 hours
AI                  : 2.00 hours
OOM                 : 5.00 hours
DEV                 : 1.00 hours
Adv Algorithm       : 5.00 hours
-----------------------------------
Total hours           : 15.00
Best fitness          : 3.2000
